# 08 — Telegram Handlers

All Telegram handlers and the ConversationHandler live in `src/handlers.py`.

## Conversation states

| Constant | Value | Description |
|---|---|---|
| `PROTOCOL_SELECT` | 0 | User is choosing protocol from inline keyboard |
| `AWAITING_OBJECTIVE` | 1 | Protocol chosen, waiting for objective text |
| `EXPERIMENT_ACTIVE` | 2 | Session in progress |
| `DEVIATION_ENTRY` | 3 | Waiting for deviation description |
| `REFINE_ENTRY` | 4 | Waiting for knowledge refinement text |
| `CONFIRM_END` | 5 | Waiting for end-of-session findings |

## Multi-user safety

`ConversationHandler` keys on `(chat_id, user_id)` by default — each researcher has fully isolated state. `concurrent_updates=False` is set in `main.py` as required.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.handlers import (
    build_conversation_handler,
    PROTOCOL_SELECT, AWAITING_OBJECTIVE, EXPERIMENT_ACTIVE,
    DEVIATION_ENTRY, REFINE_ENTRY, CONFIRM_END,
    cmd_start, cmd_help, cmd_calculate,
    fallback_text, fallback_voice, fallback_photo,
)
print('Imports OK')

## ConversationHandler structure

In [ ]:
conv = build_conversation_handler()
print('Entry points:', [type(h).__name__ for h in conv.entry_points])
print('States:')
for state, handlers in conv.states.items():
    state_name = {0:'PROTOCOL_SELECT',1:'AWAITING_OBJECTIVE',2:'EXPERIMENT_ACTIVE',
                  3:'DEVIATION_ENTRY',4:'REFINE_ENTRY',5:'CONFIRM_END'}.get(state, state)
    handler_names = [type(h).__name__ for h in handlers]
    print(f'  {state_name}: {handler_names}')

## Handler flow reference

```
/start_experiment
  → list Drive protocols (inline keyboard) → PROTOCOL_SELECT
  → user clicks protocol → ask for objective → AWAITING_OBJECTIVE
  → user types objective → load ProtocolSession → EXPERIMENT_ACTIVE

EXPERIMENT_ACTIVE
  text / voice / photo  → session.handle_message()
  /buffer [name]        → session.handle_message() with buffer prompt
  /deviation            → ask for description → DEVIATION_ENTRY
  /refine               → ask for finding → REFINE_ENTRY
  /calculate [query]    → session.handle_message() or plain Claude
  /note [text]          → session.handle_message() note prompt
  /end                  → ask for findings → CONFIRM_END

CONFIRM_END
  text (finding)        → session.handle_refine() → session.end_session() → END
  'no' / 'skip'         → session.end_session() → END

/cancel (any state)     → clear session → END
```

Outside a session (IDLE), text/voice/photo fall through to the plain fallback handlers
which use per-user `ConversationHistory` (no protocol context).